In [ ]:
from mnist import Net, test
from hyptorch.pmath import dist0

%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib import cm

import torch
from torchvision import datasets, transforms
import numpy as np
import os
import json
import dataclasses as dc

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

np.random.seed(0)
torch.manual_seed(0)

state_dict_path = "checkpoints/fashion_mnist_cnn_10.pt"
args_path = "fashion_args.json"
assert os.path.isfile(state_dict_path)
assert os.path.isfile(args_path)

In [ ]:
save_imgs = True
if save_imgs:
    os.makedirs("exp_images", exist_ok=True)

In [ ]:
with open(args_path, "r") as f:
    args = json.load(f)
args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)

In [ ]:
model = Net(args).to(device)

state_dict = torch.load(state_dict_path, map_location=device)
model.load_state_dict(state_dict)

In [ ]:
kwargs = {"num_workers": 1, "pin_memory": True} if device.type == "cuda" else {}
fashion_data_root = "../data/fashion_mnist"
train_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        fashion_data_root,
        train=True,
        download=True,
        transform=transforms.Compose(
            [transforms.ToTensor()]
        ),
    ),
    batch_size=args.batch_size,
    shuffle=True,
    **kwargs
)

test_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        fashion_data_root,
        train=False,
        transform=transforms.Compose(
            [transforms.ToTensor()]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
test(None, model, device, test_loader)

In [ ]:
data, target = next(iter(test_loader))
data, target = data.to(device), target.to(device)
output = model(data)

num_samples = 60
sample_data = data.detach().cpu().numpy()[:num_samples]
sample_target = target.detach().cpu().numpy()[:num_samples]
sample_embeddings = model.embed(data)[:num_samples]

In [ ]:
np_sample_embeddings = sample_embeddings.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 10))

circle = plt.Circle((0, 0), 1, color="black", fill=False)
ax.add_artist(circle)

colors = cm.rainbow(np.linspace(0, 1, 10))
for i, (x, y) in enumerate(np_sample_embeddings[:, :2]):
    img = sample_data[i, 0]
    ax.imshow(img, extent=(x+0.05-0.05, x+0.05+0.05, y+0.05-0.05, y+0.05+0.05), cmap="gray")
    ax.scatter(x, y, color=colors[sample_target[i]], s=50)

ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect("equal")

plt.axis("off")
if save_imgs:
    plt.savefig("exp_images/fashion_embedding_plot.png")
else:
    plt.show()

In [ ]:
x = np_sample_embeddings[:, 0]
y = np_sample_embeddings[:, 1]
z = np_sample_embeddings[:, 2]

r = np.sqrt(x ** 2 + y ** 2 + z ** 2)
theta = np.arccos(z / r)
s = np.array([1 if _y > 0 else -1 for _y in y])
phi = s * np.arccos(x / np.sqrt(x ** 2 + y ** 2))

spherical = np.zeros((r.size, 3))
spherical[:, 0] = r
spherical[:, 1] = theta
spherical[:, 2] = phi

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

colors = cm.rainbow(np.linspace(0, 1, 10))
for i, (r, theta, phi) in enumerate(spherical):
    img = sample_data[i, 0]
    ax.scatter(phi, theta, color=colors[sample_target[i]], s=50, alpha=(1-r)**(0.25))

plt.xlabel("phi")
plt.ylabel("theta")
plt.show()

In [ ]:
np.save("embeddings.npy", model.embed(data).detach().cpu().numpy())
np.save("target.npy", target)